### LCEL(LangChain Expression Language)
- LangChain 프토로콜을 사용하여 복잡한 체인(Chain)을 더 쉽고 직관적으로 구축하기 위해 설계된 선언적 언어이다.
- 여러 구성 요소(프롬프트, 모델, 출력 파싱 등)를 리눅스의 파이프 연산자(|)와 유사한 방식으로 연결하여 하나의 흐름을 만드는 방식이다.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.

1. 직항이 있다면 직항 기준 소요 시간.
2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.
3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.
"""

prompt_template = PromptTemplate.from_template(template)
prompt_template

In [ ]:
prompt = prompt_template.format(country="아이슬란드")
prompt

In [ ]:
prompt = prompt_template.format(country="몽골")
prompt

### Chian 생성

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.

1. 직항이 있다면 직항 기준 소요 시간.
2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.
3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.
"""

prompt_template = PromptTemplate.from_template(template)
prompt_template

In [ ]:
chain = prompt_template | llm

In [ ]:
chain.invoke({"country": "아이슬란드"})

In [ ]:
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)

In [ ]:
stream_response(chain.stream({"country": "몽골"}))

### load_prompt

In [ ]:
from langchain_core.prompts import load_prompt

prompt_template = load_prompt("./prompt/dog.yml", encoding="utf-8")
prompt_template.format(dog="치와와")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

chain = prompt_template | llm | StrOutputParser()

In [ ]:
chain.invoke("치와와")

### ChatPromptTemplate
- 이전 대화 내용을 기억시켜서 대화 내용이 잘 이어지게 도와주는 템플릿
- system: 시스템 설정 메시지
- human: 사용자 입력 메시지
- ai: AI의 답변 메시지

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# messages = [
#     ('system', """
#         당신은 부동산 {major} 전문 변호사입니다. 질문에 대해 간단하게 답변하며, 
#         질문이나 자세한 내용을 알고자 하지 않습니다.
#         1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.
#         2. 존댓말은 절대 사용하지 않고 반말로 답변한다.
#         3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.
#         """)
# ]

# for tag, message in DB.SELECT()
#     messages.append((tag, message))

# messages.append(("human", "{content}"))

chat_template = ChatPromptTemplate.from_messages(
    [
        ('system', """
        당신은 {major} 전문 변호사입니다. 질문에 대해 간단하게 답변하며, 
        질문이나 자세한 내용을 알고자 하지 않습니다.
        1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.
        2. 존댓말은 절대 사용하지 않고 반말로 답변한다.
        3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.
        """),
        ("human", "나 어제 땅 샀다!!"),
        ("ai", "축하해! 그 땅은 용도가 뭔데?"),
        ("human", "지목이 田이야"),
        ("ai", "올~ 그 땅에서 뭐하려고?"),
        ("human", "{content}")
    ]
)

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

chat_template.format_messages(major="부동산", content="집 지어도 돼나?")

In [ ]:
chain = chat_template | llm | StrOutputParser()
chain.invoke({"major": "부동산", "content": "집 지어도 돼나?"})

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

chat_template = ChatPromptTemplate.from_messages([
    ('system', """
        당신은 {major} 전문 변호사입니다. 질문에 대해 간단하게 답변하며, 
        질문이나 자세한 내용을 알고자 하지 않습니다.
        1. 모든 답변은 반드시 'ㅋㅋ', 'ㅋㅋㅋ', '에효..', 와 같은 이외에도 한심하다는 듯한 표현으로 시작한다.
        2. 존댓말은 절대 사용하지 않고 반말로 답변한다.
        3. 변호사처럼 길게 설명하지 말고 2문장 이내로 짧게 꾸짖는다.
        """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{content}")
])

history_data = [
    ("human", "나 어제 땅 샀다!!"),
    ("ai", "축하해! 그 땅은 용도가 뭔데?"),
    ("human", "지목이 田이야"),
    ("ai", "올~ 그 땅에서 뭐하려고?"),
]

chain = chat_template | llm | StrOutputParser()
result = chain.invoke({
    "major": "부동산", 
    "content": "집 지어도 돼나?",
    "chat_history": history_data
})
print(f'변호사 답변: {result}')

### OutputParser
- 답변을 문자열로 리턴하기 때문에 추가 작업 없이 바로 사용할 수 있다.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.

1. 직항이 있다면 직항 기준 소요 시간.
2. 직항이 없다면 가장 효율적인 경유 노선과 대기 시간을 포함한 최소 소요 시간.
3. 답변은 '소요 시간: OO시간' 형식으로만 작성해줘.
"""

prompt_template = PromptTemplate.from_template(template)

In [ ]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt_template | llm | StrOutputParser()
chain.invoke("호주")

### JsonOutputParser
- JSON 형식의 답변을 dict 자료형으로 리턴하기 때문에 추가 변환 작업 없이 바로 사용할 수 있다.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간, 직항 여부, 경유지, 경유 횟수를 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

prompt_template = PromptTemplate(
    template=template,
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

In [ ]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

In [ ]:
chain = prompt_template | llm | json_parser
chain.invoke("스위스")

### PydanticOutputParser
- JsonOutputParser보다 한 단계 더 진화한 형태이며, 사용자가 정의한 데이터 모델 규격에 맞춰서 가져올 수 있다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

# 1. 응답받고 싶은 데이터 구조 정의
class FlightInfo(BaseModel):
    country: str = Field(description="도착 국가명")
    stopovers: list = Field(description="경유지 국가명들")
    avg_hours: float = Field(description="평균 소요 시간")
    is_direct: bool = Field(description="직항 여부")
    recommendation: str = Field(description="여행객을 위한 짧은 팁")

# 2. parser에 제작한 클래스 전달
pydantic_parser = PydanticOutputParser(pydantic_object=FlightInfo)

template = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간, 직항 여부, 경유지, 경유 횟수를 알려줘.
반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
{format_instructions}
"""

# 3. 프롬프트 설정
prompt_template = PromptTemplate(
    template=template,
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

# 4. 체인 구성 및 실행
chain = prompt_template | llm | pydantic_parser
result = chain.invoke("스위스")

In [ ]:
print(type(result))
print(type(result.model_dump()))
print(type(result.model_dump_json()))

### Batch
- 여러 개의 질문을 list로 받아 일괄 처리를 수행한다

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

template = """
{stock1}, {stock2}, {stock3}에 투자할 때 각 투자 비중을 1문장으로 말해주고 3문장으로 설명해줘.
총 4문장으로만 답변하고 그 외 대답은 하지마.
"""

prompt_template = PromptTemplate.from_template(template)

chain = prompt_template | llm | StrOutputParser()
chain.batch(
    [
        {"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"},
        {"stock1": "SK하이닉스", "stock2": "S&P500", "stock3": "QQQ"},
        {"stock1": "레인보우 로보틱스", "stock2": "국채", "stock3": "적금"},
        {"stock1": "엔비디아", "stock2": "엔화", "stock3": "골드만삭스"},
    ],
    config={"max_concurrency": 3} # 동시에 처리할 수 있는 최대 작업 수
)

### Async
- 단일 쓰레드에 여러 요청이 들어오면, 요청 처리 중에는 다른 요청을 처리할 수 없기에 서버 전체가 멈출 수 있다. 따라서 비동기는 단일 쓰레드 환경에서도 여러 요청을 효율적으로 처리하고 서버의 응답성을 유지하는 핵심 기술이다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

template = """
{stock1}, {stock2}, {stock3}에 투자할 때 각 투자 비중을 1문장으로 말해주고 3문장으로 설명해줘.
총 4문장으로만 답변하고 그 외 대답은 하지마.
"""

prompt_template = PromptTemplate.from_template(template)

chain = prompt_template | llm | StrOutputParser()
response = chain.ainvoke({"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"})
await response

In [ ]:
response = chain.astream({"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"})
async for token in response:
    print(token, end="", flush=True)

In [ ]:
responses = chain.abatch(
    [
        {"stock1": "삼성전자", "stock2": "채권", "stock3": "달러"},
        {"stock1": "SK하이닉스", "stock2": "S&P500", "stock3": "QQQ"},
        {"stock1": "레인보우 로보틱스", "stock2": "국채", "stock3": "적금"},
        {"stock1": "엔비디아", "stock2": "엔화", "stock3": "골드만삭스"},
    ],
    config={"max_concurrency": 3} # 동시에 처리할 수 있는 최대 작업 수
)

await responses

### Parallel
- Batch는 하나의 template에 여러 개의 데이터를 넣지만, Parallel은 하나의 데이터를 여러 개의 template에 넣는다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

In [ ]:
from langchain_core.runnables import RunnableParallel

# template 정의
template1 = """
한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 평균 시간을 알려줘.
답변은 
 - 직항일 경우: '소요 시간: OO시간'
 - 직항이 없을 경우: '소요 시간: OO시간, 경유 : O회, 경유지: 국가명1, 국가명2,...' 
형식으로만 작성해줘.
"""

template2 = """
    {country}를 3문장으로 설명해줘
"""

prompt_template1 = PromptTemplate.from_template(template1)
prompt_template2 = PromptTemplate.from_template(template2)

chain1 = (
    prompt_template1
    | llm
    | StrOutputParser()
)

chain2 = (
    prompt_template2
    | llm
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성
combined = RunnableParallel(flight=chain1, country=chain2)

In [ ]:
response = combined.ainvoke("노르웨이")
print(await response)

In [ ]:
responses = combined.abatch(["노르웨이", "스페인"])
print(await responses)

### RunnablePassthrough
- 뒤에 오는 프롬프트가 기존 정보까지 모두 알아야 할 때, 앞의 정보가 사라지지 않게 보존하면서 새로운 정보까지 추가할 수 있다.
- LCEL의 파이프라인 구조는 한 방향으로 흘러가기 때문에, 중간에 변수 추가가 어렵다.  
  따라서 RunnablePassthrough라는 전용 통로가 필요하다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 칼로리를 계산해주는 함수
def get_calories(input_dict):
    return "800kcal"


prompt_template = PromptTemplate.from_template(
    """
    {menu}를 만들 거야. 이건 {calories} 정도 되는 요리니까 칼로리를 줄이면서도 
    맛있는 레시피를 5문장으로 작성해줘.
    마지막에는 기존 칼로리와 최종 칼로리를 말해줘. 레시피와 관련 없는 말은 하지 말아줘.
    """
)

chain = (
    RunnablePassthrough.assign(calories=get_calories) 
    | prompt_template 
    | llm 
    | StrOutputParser()
)

result = chain.ainvoke({"menu": "치즈버거"})
await result

#### 실습

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

class FlightInfo(BaseModel):
    flight_time: int = Field(description="평균 소요 시간")

async def get_flight_time(x):
    parser = PydanticOutputParser(pydantic_object=FlightInfo)
    
    template1 = """
    한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 시간으로 알려줘.
    반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
    {format_instructions}
    """
    
    prompt1 = PromptTemplate(
        template=template1,
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain1 = prompt1 | llm | parser

    resp = await chain1.ainvoke({"country": x["country"]})
    return resp.flight_time
    
template2 = """
    한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
    비용(원화)과 기내 준비물을 5문장으로 설명하라.
    앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""

prompt2 = PromptTemplate.from_template(template2)

combined_chain = (
    RunnablePassthrough.assign(
        flight_time=get_flight_time
    )
    | prompt2 
    | llm 
    | StrOutputParser()
)

result = combined_chain.ainvoke({"country": "스페인"})
print(await result)

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 객체 생성
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4.1-nano",
)

# 응답받고 싶은 데이터 구조 정의
class FlightInfo(BaseModel):
    flight_time: int = Field(description="평균 소요 시간")

async def get_flight_time(x):
    parser = PydanticOutputParser(pydantic_object=FlightInfo)
    
    template1 = """
    한국(인천 공항)에서 {country}까지 비행기를 이용할 때 소요되는 비행 시간을 시간으로 알려줘.
    반드시 JSON 형식으로만 답변하고, 앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
    {format_instructions}
    """
    
    prompt1 = PromptTemplate(
        template=template1,
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    chain1 = prompt1 | llm | parser

    resp = await chain1.ainvoke({"country": x["country"]})
    return resp.flight_time

template2 = """
    한국(인천 공항)에서 {country}까지 {flight_time}시간 걸린다.
    비용(원화)과 기내 준비물을 5문장으로 설명하라.
    앞뒤에 인사말이나 부연 설명을 절대 붙이지 마.
"""

prompt1 = PromptTemplate(
    template=template1,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
prompt2 = PromptTemplate.from_template(template2)

# 병렬 처리 레이어 구축
parallel_layer = RunnableParallel(
    # 입력된 country를 그대로 통과시킴
    country=RunnablePassthrough(), 
    flight_time=get_flight_time
)

# 전체 체인 조립
combined_chain = (
    parallel_layer  # 여기서 {"country": "스페인", "flight_time": 13} 생성됨
    | prompt2 
    | llm 
    | StrOutputParser()
)

# 실행
result = combined_chain.ainvoke({"country": "스페인"})
print(await result)

### RunnableLambda
- 외부 함수를 랭체인 내에서 실행할 수 있도록 해준다.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(
    temperature=0.2,
    model_name="gpt-4.1-nano"
)

prompt_template = PromptTemplate.from_template(
    "제목: {title}, 제목을 보고 내용 5문장을 제작하라. 앞뒤에 인사말이나 부연 설명을 붙이지마라."
)

def insert(content):
    print(f"INSERT DONE")
    print(content)
    
    return content


In [ ]:
chain = (
    prompt_template
    | llm
    | StrOutputParser()
    | RunnableLambda(insert)
)

content = chain.invoke("지구 온난화를 막는 방법")
print(content)